In [2]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from google import genai
import uvicorn
import nest_asyncio
import asyncio
import threading

# 🛠️ 1. Enable Native Notebook-to-Notebook Interoperability 
import import_ipynb
from router import AgentRouter
from faq import FAQAgent
from billing import BillingAgent

# 🛠️ 2. Apply Async Patch
nest_asyncio.apply()

app = FastAPI(title="Multi-Agent AI Hub Server")

# Instantiate and build agent modules
router = AgentRouter()
faq_agent = FAQAgent()
billing_agent = BillingAgent()

# Secure client deployment calling the Gemini 3.5 architecture
GEMINI_API_KEY = "AQ.Ab8RN6LFdv45V162pzEi3yDrxG-99IGcj98ajJAzyeW_9pj_SA"
client = genai.Client(api_key=GEMINI_API_KEY)

class QueryRequest(BaseModel):
    text: str

@app.post("/api/chat")
def chat_endpoint(request: QueryRequest):
    query = request.text
    
    # 1. Pipeline Routing Execution
    intent, confidence = router.route_query(query)
    
    context = ""
    agent_execution_log = ""
    
    # 2. Specialized Agent Path Allocation
    if intent == "faq":
        context = faq_agent.retrieve_context(query)
        agent_execution_log = "FAQ Agent retrieved structural RAG context data blocks from ChromaDB."
    elif intent == "billing":
        context = billing_agent.process_billing_action(query)
        agent_execution_log = "Billing Agent validated transaction schemas against complaints database records."
    elif intent == "technical":
        context = "System Diagnostics Guidelines: Verify client access tokens, wipe system local session caches, check routing tables."
        agent_execution_log = "Technical Agent extracted runtime troubleshooting sequences."
    else:
        context = "Standard operational system parameters apply."
        agent_execution_log = "General routing parameters initialized."

    # 3. Intelligent Generation Layer Synthesis
    try:
        prompt = f"""
        You are a highly intelligent multi-agent master customer support system. Answer the User Query accurately.
        Synthesize the specialized system background context if relevant, but use your core intelligence if the context is missing specific instructions.

        Specialized System Context: {context}
        User Query: {query}
        """
        
        # Synchronous execution is completely safe now since it runs inside an isolated thread
        response = client.models.generate_content(
            model='gemini-3.5-flash',
            contents=prompt,
        )
        ai_response = response.text
    except Exception as e:
        ai_response = f"Synthesis execution failed. Raw Engine Context: {context}. Error details: {str(e)}"

    return {
        "intent": intent,
        "confidence": float(confidence),
        "log": agent_execution_log,
        "context": context,
        "response": ai_response
    }

# 🛠️ 3. Local App Run Trigger Sequence (Thread-Isolated Jupyter Config)
if __name__ == "__main__":
    print("🚀 Starting FastAPI Multi-Agent Server on http://127.0.0.1:8001")
    print("⚠️ The cell execution will finish, but the server remains active in a background thread!")
    
    # Using Port 8001 to completely clear out any stuck hanging port 8000 processes
    config = uvicorn.Config(app, host="127.0.0.1", port=8001, log_level="info")
    server = uvicorn.Server(config)
    
    # Spin up the server completely isolated from Jupyter's loop constraints
    server_thread = threading.Thread(target=server.run, daemon=True)
    server_thread.start()

📦 Local Knowledge Base loaded successfully from disk.
🔄 Initializing Billing Agent database...
✅ Billing Agent initialized with static transaction metadata parameters.
🚀 Starting FastAPI Multi-Agent Server on http://127.0.0.1:8001
⚠️ The cell execution will finish, but the server remains active in a background thread!
